In [17]:
pip install imblearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 1.3 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 1.2 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 1.1 MB/s eta 0:01:31
   ---------------------------------------- 0.8/101.7 MB 1.1 MB/s eta 0:01:31
   ---------------------------------------- 1.0/101.7 MB 1.2 MB/s eta 0:01:24
    --------------------------------------- 1.3/101.7 MB 1.2 MB/s eta 0:01:24
    --------------------------------------- 1.6/101.7 MB 1.2 MB/s eta 0:01:24
    --------------------------------------- 1.8/101.7 MB 1.2 MB/s eta 0:01:24
    --------------------------------------- 2.1/101.7 MB 1.2 MB/s eta 0:01:23
    --------------------------------------- 2.4/101.7 MB 1.2 MB/s eta 0:01:23
   - -------------------------------------- 2.6/101.7 MB 1.2 MB/s eta 0:01:23
   - -------------------------------------- 2.9/101.7 MB 1.2 MB/s eta 0:01:22
   - -------------------------------------- 3.1/101.7 MB 1.2 MB/s eta 0:01:22



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
from sklearn.metrics import precision_recall_curve

In [11]:
train = pd.read_csv('../6.Data/Nathan_Preprocessed_train.csv')
test = pd.read_csv('../6.Data/Nathan_Preprocessed_test.csv')

target = 'target_is_fraud'
id_col = 'customer_id'

selected_features = [col for col in train.columns if col not in [target, id_col]]

X = train[selected_features]
y = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scale = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr'
)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
}

scorer = make_scorer(f1_score, pos_label=1)

grid = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print('Best params:', grid.best_params_)

# Optimisation du seuil
y_proba = best_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[f1_scores.argmax()]
print(f"Meilleur seuil : {best_threshold:.3f}")

y_pred = (y_proba >= best_threshold).astype(int)

print('\nConfusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Meilleur seuil : 0.734

Confusion Matrix:
[[27061  2184]
 [ 1330  1425]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94     29245
           1       0.39      0.52      0.45      2755

    accuracy                           0.89     32000
   macro avg       0.67      0.72      0.69     32000
weighted avg       0.91      0.89      0.90     32000

